In [8]:
import json
import pandas as pd

# import top 25 CWEs
with open("../data/raw/top_25_cwe_data.json", "r", encoding="utf-8") as f:
    cwe_data = json.load(f)

TOP_25_CWES = cwe_data["TOP_25_CWES"]

# build CWE map for TOSEM
with open("../data/raw/tool_assisted_manual_dataset.json") as f:
    dataset = json.load(f)

rows = []
for entry in dataset:
    cwe = entry.get("cwe", "Unknown")
    for commit in entry.get("fixing", []):
        rows.append({"commit_id": commit, "cwe": cwe})
    intro = entry.get("introducing")
    if intro:
        rows.append({"commit_id": intro, "cwe": cwe})

df_cwe_map = pd.DataFrame(rows)

# merge to get CWEs
df_tosem = pd.read_csv("../data/intermediate/churn_tosem.csv")
df_tosem = df_tosem.merge(df_cwe_map, on="commit_id", how="left")

df_top25 = df_tosem[df_tosem["cwe"].isin(TOP_25_CWES)]

print(len(df_top25))
print(len(df_tosem))

158
281


In [9]:
import pandas as pd

# Load both
df_icvul = pd.read_csv("../data/intermediate/churn_icvul.csv")
df_icvul_raw = pd.read_csv("../data/raw/icvul.csv")

# Build commit_id → cwe_id lookup from fc_hash only
commit_to_cwe = df_icvul_raw[["fc_hash", "cwe_id"]].rename(columns={"fc_hash": "commit_id"}).drop_duplicates(subset="commit_id")

# Merge into processed ICVul df
df_icvul = df_icvul.merge(commit_to_cwe, on="commit_id", how="left")

# Filter to top 25
df_icvul_filtered = df_icvul[df_icvul["cwe_id"].isin(TOP_25_CWES)]

print(f"Total entries after filtering: {len(df_icvul_filtered)}")
print(f"CWEs represented: {df_icvul_filtered['cwe_id'].nunique()}")
print(df_icvul_filtered["cwe_id"].value_counts())

Total entries after filtering: 133
CWEs represented: 12
cwe_id
CWE-125    46
CWE-416    20
CWE-787    16
CWE-476    16
CWE-20     15
CWE-200    10
CWE-122     3
CWE-120     3
CWE-22      1
CWE-284     1
CWE-770     1
CWE-77      1
Name: count, dtype: int64
